# 📄 DocBrain RAG - Google Colab
Upload PDF/DOCX/TXT files and ask questions using Gemini + FAISS + Sentence Transformers.

In [ ]:
# Install dependencies
!pip -q install sentence-transformers faiss-cpu pymupdf python-docx langchain google-generativeai ipywidgets

In [ ]:
import os
import fitz
import faiss
import numpy as np
import google.generativeai as genai
from google.colab import files
from docx import Document
from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter

GOOGLE_API_KEY = input("Enter Gemini API Key: ")
genai.configure(api_key=GOOGLE_API_KEY)
llm = genai.GenerativeModel("gemini-2.5-flash")
embedder = SentenceTransformer("all-MiniLM-L6-v2")


In [ ]:
uploaded = files.upload()

In [ ]:
def read_pdf(path):
    doc = fitz.open(path)
    txt=""
    for p in doc:
        txt += p.get_text()
    return txt

def read_docx(path):
    d = Document(path)
    return "\n".join(p.text for p in d.paragraphs)

def read_txt(path):
    with open(path,"r",encoding="utf-8",errors="ignore") as f:
        return f.read()

texts=[]
for f in uploaded.keys():
    if f.endswith(".pdf"):
        texts.append(read_pdf(f))
    elif f.endswith(".docx"):
        texts.append(read_docx(f))
    elif f.endswith(".txt"):
        texts.append(read_txt(f))

text="\n".join(texts)
print("Characters:",len(text))


In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks=splitter.split_text(text)
print("Chunks:",len(chunks))
embeddings=embedder.encode(chunks,convert_to_numpy=True).astype("float32")

index=faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print("Indexed:",index.ntotal)


In [ ]:
def ask(question,k=5):
    q=embedder.encode([question],convert_to_numpy=True).astype("float32")
    D,I=index.search(q,k)
    context="\n\n".join(chunks[i] for i in I[0])

    prompt=f'''
Answer ONLY from the supplied context.
If the answer is unavailable, say:
"I couldn't find that in the uploaded documents."

Context:
{context}

Question:
{question}
'''

    response=llm.generate_content(prompt)
    return response.text


In [ ]:
while True:
    q=input("Question (type exit): ")
    if q.lower()=="exit":
        break
    print("\nAnswer:\n")
    print(ask(q))
    print("-"*80)
